# EchoVerify: executed training, evaluation, and placement experiment

This notebook runs a real integration training pass on a small subset, examines the saved three-epoch baseline runs, recomputes full held-out metrics from saved predictions, and executes the paired background-placement audit. The full baseline weights were trained on HiPerGator before this notebook. The notebook integration checkpoint is stored separately.

Run from the EchoVerify repository using the `echoverify` kernel. Set `ASVSPOOF2019_LA_ROOT` to the extracted LA folder and `EV_ROOT` to the folder containing the saved `checkpoints/` and `results/`. All reported numbers below are calculated when the cells execute.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, Image, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
ARTIFACT_ROOT = Path(os.environ.get("EV_ROOT", REPO_ROOT / "EchoVerify_run_outputs"))
DATA_ROOT = Path(os.environ["ASVSPOOF2019_LA_ROOT"])
OUTPUT = ARTIFACT_ROOT / "results" / "notebook_evidence"
OUTPUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from src.data.protocol import ASVSpoofProtocol
from src.evaluation.metrics import compute_all_metrics
from src.predict import EchoVerifyPredictor

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Data root:", DATA_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
assert (REPO_ROOT / "src" / "placement_experiment.py").exists()

## 1. Load the real benchmark and verify the original cohorts

Only training data fit weights. Development data selected checkpoints and operating thresholds. The official Eval partition was scored after those choices were fixed.

In [ ]:
protocol = ASVSpoofProtocol(DATA_ROOT)
partitions = {name: protocol.load_split(name) for name in ("train", "dev", "eval")}
counts = pd.DataFrame([
    {"split": name, "genuine": int((df.label == 0).sum()),
     "synthetic": int((df.label == 1).sum()), "total": len(df)}
    for name, df in partitions.items()
])
display(counts)
for split in ("train", "dev"):
    raw = pd.read_csv(ARTIFACT_ROOT / "results" / "raw" / f"manifest_{split}.csv")
    trimmed = pd.read_csv(ARTIFACT_ROOT / "results" / "trimmed" / f"manifest_{split}.csv")
    assert raw.utt_id.tolist() == trimmed.utt_id.tolist()
    print(split, "paired recording IDs verified:", len(raw))
assert set(partitions["train"].utt_id).isdisjoint(partitions["dev"].utt_id)
assert set(partitions["train"].utt_id).isdisjoint(partitions["eval"].utt_id)
assert set(partitions["dev"].utt_id).isdisjoint(partitions["eval"].utt_id)

## 2. Execute a small training run on actual audio

This one-epoch integration run uses 24 training recordings and 16 development recordings. Its purpose is to demonstrate that training executes within the notebook. It does not replace the saved 2,000/1,000 baseline experiments or establish benchmark performance.

In [ ]:
command = [sys.executable, str(REPO_ROOT / "src" / "train.py"),
           "--data-root", str(DATA_ROOT), "--smoke-run", "--device", DEVICE,
           "--num-workers", "0", "--output-dir", str(OUTPUT / "integration_training"),
           "--checkpoint-dir", str(OUTPUT / "integration_checkpoint")]
run = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
log = run.stdout + run.stderr
(OUTPUT / "integration_training.log").write_text(log, encoding="utf-8")
print(log)
run.check_returncode()
assert (OUTPUT / "integration_checkpoint" / "best_model.pt").exists()
print("Real-data integration training completed and checkpoint saved.")

## 3. Inspect the completed three-epoch baseline runs

These histories were generated by the earlier HiPerGator runs using 2,000 training clips and 1,000 development clips. They are loaded from their saved artifacts.

In [ ]:
development = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for mode in ("raw", "trimmed"):
    folder = ARTIFACT_ROOT / "results" / mode
    history = json.loads((folder / "history.json").read_text())
    config = json.loads((folder / "config.json").read_text())
    metrics = json.loads((folder / "dev_metrics.json").read_text())
    print(mode, "training configuration:", config["args"])
    development.append({"model": mode, "split": "dev", **metrics})
    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label=f"{mode} train")
    axes[0].plot(history["epoch"], history["dev_loss"], marker="x", label=f"{mode} dev")
    axes[1].plot(history["epoch"], history["dev_eer"], marker="o", label=mode)
axes[0].set(xlabel="Epoch", ylabel="BCE loss")
axes[1].set(xlabel="Epoch", ylabel="Development EER")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT / "baseline_training_curves.png", dpi=160)
plt.show()
columns = ["model", "split", "eer", "roc_auc", "accuracy", "genuine_speech_fpr", "spoof_fnr"]
display(pd.DataFrame(development)[columns])
baseline = json.loads((ARTIFACT_ROOT / "results" / "silence_baseline" / "silence_baseline_dev_metrics.json").read_text())
print("Silence-duration-only development results:")
display(pd.DataFrame([baseline])[["eer", "roc_auc", "accuracy", "genuine_speech_fpr"]])

## 4. Recompute the full held-out metrics from saved per-recording scores

The fixed thresholds are loaded from the original evaluation records. Recomputing here checks the exported evidence without selecting a new threshold or retraining on Eval.

In [ ]:
heldout = []
for mode in ("raw", "trimmed"):
    folder = ARTIFACT_ROOT / "results" / mode / "heldout"
    record = json.loads((folder / "eval_metrics.json").read_text())
    scores = pd.read_csv(folder / "eval_scores.csv")
    assert len(scores) == 71237
    assert set(scores.utt_id) == set(partitions["eval"].utt_id)
    metrics = compute_all_metrics(scores.true_label.to_numpy(), scores.score.to_numpy(),
                                  threshold=record["applied_fixed_threshold"])
    assert metrics["confusion_matrix"] == record["metrics"]["confusion_matrix"]
    heldout.append({"model": mode, "split": "eval", **metrics})
    print(mode, "fixed threshold:", record["applied_fixed_threshold"])
    display(Image(filename=str(folder / "confusion_matrix_eval.png"), width=440))
heldout_table = pd.DataFrame(heldout)[columns]
display(heldout_table)
heldout_table.to_csv(OUTPUT / "verified_heldout_metrics.csv", index=False)

## 5. Run real-file inference with each saved model

These examples are selected deterministically by their known labels. Prediction errors are retained. Two examples demonstrate execution and do not estimate generalization.

In [ ]:
examples = []
for mode in ("raw", "trimmed"):
    predictor = EchoVerifyPredictor(ARTIFACT_ROOT / "checkpoints" / mode / "best_model.pt", device=DEVICE)
    for label in (0, 1):
        row = partitions["eval"][partitions["eval"].label == label].sort_values("utt_id").iloc[0]
        result = predictor.predict(row.audio_path)
        examples.append({"model": mode, "utt_id": row.utt_id,
                         "true_label": label, "prediction": result["prediction"],
                         "score": result["raw_score"], "threshold": result["operating_threshold"]})
display(pd.DataFrame(examples))
pd.DataFrame(examples).to_csv(OUTPUT / "sample_predictions.csv", index=False)

## 6. Execute the controlled placement experiment

For 100 genuine and 100 synthetic Eval recordings with at least three seconds retained after edge trimming, reuse the exact same three-second excerpt in all six conditions. Split one second of background across the beginning and end at 0.1/0.9, 0.5/0.5, and 0.9/0.1 seconds. Compare zero-valued padding with recorded low-energy edges from training bona fide audio, scaled to -50 dBFS RMS.

The script scores constructed four-second inputs directly through the frozen models. It bypasses trimming and cropping during this audit, because those operations would remove the manipulation being measured. Model names identify how the weights were trained. Fixed development thresholds are retained.

The balanced audit cohort and automatic noise selection are documented separately from the full held-out benchmark. Saved donor recordings should be listened to for quality review.

In [ ]:
placement_folder = ARTIFACT_ROOT / "results" / "placement"
command = [sys.executable, "-m", "src.placement_experiment",
           "--data-root", str(DATA_ROOT), "--artifact-root", str(ARTIFACT_ROOT),
           "--output-dir", str(placement_folder), "--per-class", "100",
           "--noise-donors", "5", "--noise-dbfs", "-50", "--seed", "42", "--device", DEVICE]
run = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
log = run.stdout + run.stderr
(OUTPUT / "placement_run.log").write_text(log, encoding="utf-8")
print(log)
run.check_returncode()
summary = pd.read_csv(placement_folder / "placement_summary.csv")
display(summary)
display(Image(filename=str(placement_folder / "placement_flip_rates.png")))

## 7. Inspect the actual background donors

These are automatically selected training-set edge segments. Listen for speech contamination. Retain and document the run settings. Any later revised selection should be reported as a separate experiment.

In [ ]:
donors = pd.read_csv(placement_folder / "noise_donors.csv")
display(donors)
for row in donors.itertuples():
    print("Training donor", row.utt_id, row.side, "at -50 dBFS RMS")
    display(Audio(filename=str(placement_folder / f"noise_donor_{row.noise_id}.wav"), normalize=False))
print("Evidence saved under", OUTPUT)
print("Placement results saved under", placement_folder)

## Interpretation and limits

Use the printed score changes and flip rates to discuss sensitivity to background placement and background type. Inspect genuine-speech errors separately. A model that changes its prediction less may still have worse detection performance, so interpret sensitivity together with the full held-out metrics. The audit only covers clips meeting the retained-duration criterion, one background level, a small donor set, and one training seed. Accent or impairment-specific fairness cannot be measured without suitable labels.